In [1]:
from IPython.display import display

import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("./"))
import helpers as _

In [2]:
df = pd.read_csv('Meteo/data/meteo_horaire_69_69029001_1920-2023.csv')

df_sorted = df[sorted(df.columns)]
df_sorted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911664 entries, 0 to 911663
Columns: 102 entries, B1 to WW
dtypes: float64(98), int64(3), object(1)
memory usage: 709.5+ MB


In [3]:
columns_date = ['DATE']
columns_continue = ['T', 'PSTAT', 'PMER', 'TSV', 'RR1', 'DRR1', 'U', 'FF', 'N']
columns_discret = ['SOL']
mask = ~(df['T'].isnull()) & ~(df['PSTAT'].isnull()) & ~(df['PMER'].isnull()) & ~(df['TSV'].isnull()) & ~(df['RR1'].isnull()) & ~(df['DRR1'].isnull()) & ~(df['U'].isnull()) & ~(df['FF'].isnull()) & ~(df['N'].isnull()) & ~(df['SOL'].isnull())
df_filtered = df.loc[mask,columns_date+columns_continue+columns_discret]

df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 84809 entries, 18 to 298026
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DATE    84809 non-null  object 
 1   T       84809 non-null  float64
 2   PSTAT   84809 non-null  float64
 3   PMER    84809 non-null  float64
 4   TSV     84809 non-null  float64
 5   RR1     84809 non-null  float64
 6   DRR1    84809 non-null  float64
 7   U       84809 non-null  float64
 8   FF      84809 non-null  float64
 9   N       84809 non-null  float64
 10  SOL     84809 non-null  float64
dtypes: float64(10), object(1)
memory usage: 7.8+ MB


In [7]:
df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'])
df_filtered['HOUR'] = df_filtered['DATE'].dt.strftime('%H:%M:%S')
df_filtered['DAY'] = pd.to_datetime(df_filtered['DATE'].dt.strftime('%Y-%m-%d'), format='%Y-%m-%d')

df_grouped = df_filtered.set_index('DATE').resample('D')[columns_continue].median().reset_index()

mask = ~(df_grouped['T'].isnull()) & ~(df_grouped['PSTAT'].isnull()) & ~(df_grouped['PMER'].isnull()) & ~(df_grouped['TSV'].isnull()) & ~(df_grouped['RR1'].isnull()) & ~(df_grouped['DRR1'].isnull()) & ~(df_grouped['U'].isnull()) & ~(df_grouped['FF'].isnull()) & ~(df_grouped['N'].isnull())
df_grouped = df_grouped.loc[mask]

df_grouped.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10558 entries, 0 to 12115
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   DATE    10558 non-null  datetime64[ns]
 1   T       10558 non-null  float64       
 2   PSTAT   10558 non-null  float64       
 3   PMER    10558 non-null  float64       
 4   TSV     10558 non-null  float64       
 5   RR1     10558 non-null  float64       
 6   DRR1    10558 non-null  float64       
 7   U       10558 non-null  float64       
 8   FF      10558 non-null  float64       
 9   N       10558 non-null  float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 907.3 KB


In [21]:
display(
    _.generate_box([
        _.generate_widget_table(df_filtered),
        _.generate_widget_table(df_grouped),
    ])
)

Box(children=(Output(), Output()), layout=Layout(align_items='center', display='flex', flex_flow='row wrap', j…

In [22]:
display(
    _.generate_box(
        children=[
            _.generate_widget_corr(df_filtered[columns_date+columns_continue], title='By hour', width=950, height=1000),
            _.generate_widget_corr(df_grouped[columns_date+columns_continue], title='By day', width=950, height=1000)
        ]
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

In [8]:
from statsmodels.tsa.api import ExponentialSmoothing

forecaster = ExponentialSmoothing(df_filtered, trend="add", seasonal="add")


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).